# 🧪 Chemicals Pricing Intelligence
## Econometric Framework for Price Elasticity Estimation & Portfolio Optimization

---

**Author:** Chemicals Pricing Analytics Team  
**Platform:** Snowflake ML  
**Version:** 1.0

---

### Abstract

This workbook implements a comprehensive econometric framework for estimating price elasticities and optimizing pricing decisions in the chemicals industry. We employ:

1. **Ordinary Least Squares (OLS)** for baseline own-price elasticity estimation
2. **Seemingly Unrelated Regressions (SUR)** for capturing cross-price effects and substitution patterns
3. **Non-Linear Programming (NLP)** for constrained profit maximization

The methodology follows established practices in industrial organization economics and revenue management literature.

---
## Table of Contents

1. [System Architecture](#architecture)
2. [Theoretical Framework](#theory)
3. [Data Pipeline](#data)
4. [OLS Elasticity Estimation](#ols)
5. [SUR Cross-Elasticity Model](#sur)
6. [Optimization Engine](#optimization)
7. [Model Diagnostics & Validation](#diagnostics)
8. [Results & Recommendations](#results)

---
<a id='architecture'></a>
# 1. System Architecture

## 1.1 End-to-End ML Pipeline

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        CHEMICALS PRICING ML PIPELINE                        │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌───────────────┐    ┌───────────────┐    ┌───────────────┐               │
│  │  DATA LAYER   │───▶│  MODEL LAYER  │───▶│  APP LAYER    │               │
│  └───────────────┘    └───────────────┘    └───────────────┘               │
│         │                    │                    │                        │
│         ▼                    ▼                    ▼                        │
│  ┌─────────────┐      ┌─────────────┐      ┌─────────────┐                 │
│  │ Snowflake   │      │ Snowpark    │      │ Streamlit   │                 │
│  │ Tables/Views│      │ ML Pipeline │      │ Dashboard   │                 │
│  └─────────────┘      └─────────────┘      └─────────────┘                 │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

## 1.2 Data Architecture (Medallion Pattern)

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                                                                              │
│   BRONZE (RAW)              SILVER (ATOMIC)           GOLD (MART)           │
│   ────────────              ───────────────           ───────────           │
│                                                                              │
│   ┌──────────┐              ┌──────────┐              ┌──────────────────┐  │
│   │ERP_SALES │─────────────▶│SALES_    │─────────────▶│ML_TRAINING_DATA  │  │
│   │_ORDERS   │              │ORDER     │              │                  │  │
│   └──────────┘              └──────────┘              │• ln(Q), ln(P)    │  │
│                                                       │• Lagged prices   │  │
│   ┌──────────┐              ┌──────────┐              │• Market indices  │  │
│   │MARKET_   │─────────────▶│PRODUCT   │              │• Temporal feats  │  │
│   │FEEDSTOCK │              │CUSTOMER  │              └──────────────────┘  │
│   └──────────┘              └──────────┘                       │            │
│                                                                 ▼            │
│   ┌──────────┐              ┌──────────┐              ┌──────────────────┐  │
│   │PLANT_    │─────────────▶│COST_TO_  │─────────────▶│ELASTICITY_MATRIX │  │
│   │COSTS     │              │SERVE     │              │OPTIMAL_PRICING   │  │
│   └──────────┘              └──────────┘              └──────────────────┘  │
│                                                                              │
│   ╔══════════════════════════════════════════════════════════════════════╗  │
│   ║  MARKETPLACE INTEGRATION                                             ║  │
│   ║  • Fuel Oil CPI (BLS)  • Core CPI  • Industrial Production Index    ║  │
│   ╚══════════════════════════════════════════════════════════════════════╝  │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
```

## 1.3 Model Pipeline Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          MODEL ESTIMATION PIPELINE                          │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  STAGE 1: DATA PREPARATION                                                 │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Raw Data → Log Transform → Feature Engineering → Train/Test Split  │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│                                    │                                       │
│                                    ▼                                       │
│  STAGE 2: BASELINE ESTIMATION (OLS)                                        │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  For each product i:                                                 │   │
│  │    ln(Qᵢ) = αᵢ + βᵢ·ln(Pᵢ) + γᵢ·CPI + δᵢ·IP + Σλₘ·Monthₘ + εᵢ     │   │
│  │                                                                      │   │
│  │  Output: Per-product elasticity estimates {β̂ᵢ}                      │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│                                    │                                       │
│                                    ▼                                       │
│  STAGE 3: SYSTEM ESTIMATION (SUR)                                          │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Joint estimation of n demand equations:                             │   │
│  │    ln(Q₁) = α₁ + β₁₁·ln(P₁) + β₁₂·ln(P₂) + ... + ε₁                │   │
│  │    ln(Q₂) = α₂ + β₂₁·ln(P₁) + β₂₂·ln(P₂) + ... + ε₂                │   │
│  │    ...                                                               │   │
│  │                                                                      │   │
│  │  Output: n×n Elasticity Matrix E = [εᵢⱼ]                            │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│                                    │                                       │
│                                    ▼                                       │
│  STAGE 4: OPTIMIZATION                                                     │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  max  Π(P) = Σᵢ (Pᵢ - Cᵢ)·Qᵢ(P)                                     │   │
│  │   P                                                                  │   │
│  │  s.t. Margin ≥ 15%, |ΔP| ≤ 10%                                      │   │
│  │                                                                      │   │
│  │  Output: Optimal price vector P*                                     │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

---
<a id='theory'></a>
# 2. Theoretical Framework

## 2.1 Demand Theory & Price Elasticity

### Definition

**Price elasticity of demand** measures the responsiveness of quantity demanded to price changes:

$$\varepsilon = \frac{\partial Q}{\partial P} \cdot \frac{P}{Q} = \frac{\% \Delta Q}{\% \Delta P}$$

### Interpretation

| Elasticity | Classification | Revenue Effect of Price ↑ |
|------------|----------------|---------------------------|
| $\|\varepsilon\| > 1$ | **Elastic** | Revenue ↓ |
| $\|\varepsilon\| = 1$ | **Unit Elastic** | Revenue unchanged |
| $\|\varepsilon\| < 1$ | **Inelastic** | Revenue ↑ |

## 2.2 Log-Linear Demand Specification

We adopt the **constant elasticity** (log-log) functional form:

$$Q = A \cdot P^{\beta} \cdot e^{\gamma X}$$

Taking natural logarithms:

$$\ln(Q) = \underbrace{\ln(A)}_{\alpha} + \beta \cdot \ln(P) + \gamma \cdot X$$

### Why Log-Log?

**Theorem:** In the log-log specification, the coefficient β equals the price elasticity.

**Proof:**
$$\varepsilon = \frac{\partial Q}{\partial P} \cdot \frac{P}{Q}$$

From $Q = A \cdot P^{\beta}$:
$$\frac{\partial Q}{\partial P} = A \cdot \beta \cdot P^{\beta - 1} = \beta \cdot \frac{Q}{P}$$

Therefore:
$$\varepsilon = \beta \cdot \frac{Q}{P} \cdot \frac{P}{Q} = \beta \quad \blacksquare$$

## 2.3 Cross-Price Elasticity & Substitution

### Definition

**Cross-price elasticity** measures how demand for product $i$ responds to price changes in product $j$:

$$\varepsilon_{ij} = \frac{\partial Q_i}{\partial P_j} \cdot \frac{P_j}{Q_i} = \frac{\% \Delta Q_i}{\% \Delta P_j}$$

### Interpretation

| Cross-Elasticity | Relationship | Example |
|------------------|--------------|--------|
| $\varepsilon_{ij} > 0$ | **Substitutes** | PE ↔ PP |
| $\varepsilon_{ij} < 0$ | **Complements** | Resin ↔ Additive |
| $\varepsilon_{ij} \approx 0$ | **Independent** | PE ↔ Benzene |

### The Elasticity Matrix

For $n$ products, we define the **elasticity matrix** $\mathbf{E} \in \mathbb{R}^{n \times n}$:

$$\mathbf{E} = \begin{bmatrix} \varepsilon_{11} & \varepsilon_{12} & \cdots & \varepsilon_{1n} \\ \varepsilon_{21} & \varepsilon_{22} & \cdots & \varepsilon_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ \varepsilon_{n1} & \varepsilon_{n2} & \cdots & \varepsilon_{nn} \end{bmatrix}$$

**Properties:**
- Diagonal elements $\varepsilon_{ii}$ are own-price elasticities (typically negative)
- Off-diagonal elements $\varepsilon_{ij}$ are cross-price elasticities
- Matrix is generally **not symmetric** (asymmetric substitution patterns)

## 2.4 Seemingly Unrelated Regressions (SUR)

### Model Specification

The SUR model estimates a **system of equations** where error terms may be correlated:

$$\begin{aligned}
\mathbf{y}_1 &= \mathbf{X}_1 \boldsymbol{\beta}_1 + \boldsymbol{\varepsilon}_1 \\
\mathbf{y}_2 &= \mathbf{X}_2 \boldsymbol{\beta}_2 + \boldsymbol{\varepsilon}_2 \\
&\vdots \\
\mathbf{y}_n &= \mathbf{X}_n \boldsymbol{\beta}_n + \boldsymbol{\varepsilon}_n
\end{aligned}$$

### Stacked Representation

$$\mathbf{y} = \mathbf{X} \boldsymbol{\beta} + \boldsymbol{\varepsilon}$$

where:
$$\mathbf{y} = \begin{bmatrix} \mathbf{y}_1 \\ \vdots \\ \mathbf{y}_n \end{bmatrix}, \quad
\mathbf{X} = \begin{bmatrix} \mathbf{X}_1 & & \\ & \ddots & \\ & & \mathbf{X}_n \end{bmatrix}, \quad
\boldsymbol{\beta} = \begin{bmatrix} \boldsymbol{\beta}_1 \\ \vdots \\ \boldsymbol{\beta}_n \end{bmatrix}$$

### Error Structure

$$E[\boldsymbol{\varepsilon}\boldsymbol{\varepsilon}'] = \boldsymbol{\Sigma} \otimes \mathbf{I}_T$$

where $\boldsymbol{\Sigma}$ is the $n \times n$ covariance matrix of contemporaneous errors.

### GLS Estimation

The **Feasible GLS (FGLS)** estimator:

$$\hat{\boldsymbol{\beta}}_{FGLS} = (\mathbf{X}'(\hat{\boldsymbol{\Sigma}}^{-1} \otimes \mathbf{I}_T)\mathbf{X})^{-1} \mathbf{X}'(\hat{\boldsymbol{\Sigma}}^{-1} \otimes \mathbf{I}_T)\mathbf{y}$$

**Key Insight:** SUR is more efficient than equation-by-equation OLS when:
1. Error terms are correlated across equations ($\sigma_{ij} \neq 0$)
2. Regressors differ across equations

## 2.5 Optimization Theory

### Profit Maximization Problem

$$\max_{\mathbf{P} \in \mathbb{R}^n_+} \quad \Pi(\mathbf{P}) = \sum_{i=1}^{n} (P_i - C_i) \cdot Q_i(\mathbf{P})$$

### Demand Function with Cross-Effects

Using the elasticity matrix:

$$Q_i(\mathbf{P}) = Q_i^0 \cdot \exp\left( \sum_{j=1}^{n} \varepsilon_{ij} \cdot \ln\frac{P_j}{P_j^0} \right)$$

Or in matrix notation:

$$\mathbf{Q}(\mathbf{P}) = \mathbf{Q}^0 \odot \exp\left( \mathbf{E} \cdot \ln\frac{\mathbf{P}}{\mathbf{P}^0} \right)$$

### Lagrangian Formulation

With constraints, we form the Lagrangian:

$$\mathcal{L}(\mathbf{P}, \boldsymbol{\lambda}, \boldsymbol{\mu}) = -\Pi(\mathbf{P}) + \sum_i \lambda_i \cdot g_i(\mathbf{P}) + \sum_j \mu_j \cdot h_j(\mathbf{P})$$

### First-Order Conditions (KKT)

$$\nabla_{\mathbf{P}} \mathcal{L} = \mathbf{0}$$
$$\lambda_i \cdot g_i(\mathbf{P}^*) = 0 \quad \forall i \text{ (complementary slackness)}$$
$$\lambda_i \geq 0, \quad g_i(\mathbf{P}^*) \leq 0$$

### Gradient of Profit Function

$$\frac{\partial \Pi}{\partial P_k} = Q_k + \sum_{i=1}^{n} (P_i - C_i) \cdot Q_i \cdot \frac{\varepsilon_{ik}}{P_k}$$

**Interpretation:** The marginal profit from raising $P_k$ includes:
1. Direct effect on product $k$ revenue
2. Cross-effects on all other products via substitution

---
<a id='data'></a>
# 3. Data Pipeline

## 3.1 Environment Setup

In [ ]:
# ============================================================================
# IMPORTS & SESSION
# ============================================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from scipy.optimize import minimize
from scipy import stats
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
session = get_active_session()

print('=' * 60)
print('CHEMICALS PRICING ML WORKBOOK')
print('=' * 60)
print(f'Session initialized: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Snowflake version: {session.sql("SELECT CURRENT_VERSION()").collect()[0][0]}')

## 3.2 Data Ingestion

In [ ]:
# ============================================================================
# LOAD TRAINING DATA
# ============================================================================
df = session.sql('''
    SELECT *
    FROM CHEMICALS_DB.CHEMICAL_OPS.ML_TRAINING_DATA
    ORDER BY order_date, product_id
''').to_pandas()

print('\n📊 DATASET SUMMARY')
print('-' * 40)
print(f'Observations:      {len(df):>12,}')
print(f'Date Range:        {df["ORDER_DATE"].min()} to {df["ORDER_DATE"].max()}')
print(f'Products:          {df["PRODUCT_ID"].nunique():>12}')
print(f'Product Families:  {df["PRODUCT_FAMILY"].nunique():>12}')
print(f'Regions:           {df["REGION"].nunique():>12}')

print('\n📋 PRODUCT FAMILIES')
print(df.groupby('PRODUCT_FAMILY').agg(
    Products=('PRODUCT_ID', 'nunique'),
    Observations=('ORDER_DATE', 'count'),
    Avg_Price=('AVG_PRICE_PER_MT', 'mean'),
    Avg_Margin=('MARGIN_PCT', 'mean')
).round(2))

## 3.3 Feature Overview

| Feature | Formula | Description |
|---------|---------|-------------|
| `LN_QUANTITY` | $\ln(Q_{it})$ | Log of daily quantity sold |
| `LN_PRICE` | $\ln(P_{it})$ | Log of average selling price |
| `PRICE_AVG_30D` | $\bar{P}_{i,t-30:t-1}$ | 30-day trailing average price |
| `PRICE_VS_AVG_30D` | $P_{it} / \bar{P}_{30}$ | Price relative to recent average |
| `FUEL_OIL_CPI` | $CPI_t^{fuel}$ | BLS Fuel Oil Consumer Price Index |
| `INDUSTRIAL_PRODUCTION` | $IP_t$ | Fed Industrial Production Index |
| `MARGIN_PCT` | $(P - C) / P$ | Gross margin percentage |

In [ ]:
# ============================================================================
# DESCRIPTIVE STATISTICS
# ============================================================================
key_vars = ['LN_QUANTITY', 'LN_PRICE', 'MARGIN_PCT', 'FUEL_OIL_CPI', 'INDUSTRIAL_PRODUCTION']

print('\n📈 DESCRIPTIVE STATISTICS')
print(df[key_vars].describe().round(4).T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

In [ ]:
# ============================================================================
# CORRELATION ANALYSIS
# ============================================================================
corr_vars = ['LN_QUANTITY', 'LN_PRICE', 'FUEL_OIL_CPI', 'CORE_CPI', 'INDUSTRIAL_PRODUCTION', 'MARGIN_PCT']
corr_labels = ['ln(Q)', 'ln(P)', 'Fuel CPI', 'Core CPI', 'Ind. Prod.', 'Margin']
corr = df[corr_vars].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr_labels, y=corr_labels,
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=corr.round(3).values, texttemplate='%{text}',
    textfont={'size': 11}, colorbar=dict(title='ρ')
))
fig.update_layout(
    title='<b>Correlation Matrix</b><br><sup>Pearson correlation coefficients</sup>',
    template='plotly_dark', height=450, width=550
)
fig.show()

# Key correlation test
r, p = stats.pearsonr(df['LN_PRICE'].dropna(), df['LN_QUANTITY'].dropna())
print(f'\n🔍 Price-Quantity Correlation: ρ = {r:.4f} (p-value = {p:.2e})')

In [ ]:
# ============================================================================
# VISUALIZATION: PRICE-QUANTITY RELATIONSHIP
# ============================================================================
fig = px.scatter(
    df.dropna(subset=['LN_PRICE', 'LN_QUANTITY']).sample(min(5000, len(df))),
    x='LN_PRICE', y='LN_QUANTITY', color='PRODUCT_FAMILY',
    opacity=0.4, trendline='ols',
    title='<b>Demand Curve: Log-Log Specification</b><br><sup>ln(Q) vs ln(P) by product family</sup>',
    labels={'LN_PRICE': 'ln(Price per MT)', 'LN_QUANTITY': 'ln(Quantity MT)'}
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

---
<a id='ols'></a>
# 4. OLS Elasticity Estimation

## 4.1 Model Specification

For each product $i$, we estimate:

$$\ln(Q_{it}) = \alpha_i + \beta_i \ln(P_{it}) + \gamma_i \cdot CPI_t^{(standardized)} + \delta_i \cdot IP_t^{(standardized)} + \sum_{m=2}^{12} \lambda_{im} \cdot \mathbb{1}_{Month=m} + \varepsilon_{it}$$

**Assumptions:**
1. $E[\varepsilon_{it}] = 0$ (zero conditional mean)
2. $Var(\varepsilon_{it}) = \sigma_i^2$ (homoskedasticity)
3. $Cov(\varepsilon_{it}, \varepsilon_{is}) = 0$ for $t \neq s$ (no serial correlation)
4. $Cov(\varepsilon_{it}, \ln(P_{it})) = 0$ (exogeneity)

In [ ]:
# ============================================================================
# OLS ESTIMATION FUNCTION
# ============================================================================
def estimate_ols_elasticity(data, product_id, min_obs=50):
    """
    Estimate own-price elasticity using OLS with controls.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Training data with required columns
    product_id : str
        Product identifier
    min_obs : int
        Minimum observations required
    
    Returns:
    --------
    dict : Estimation results including coefficients and diagnostics
    """
    pdata = data[data['PRODUCT_ID'] == product_id].copy()
    
    if len(pdata) < min_obs:
        return None
    
    # Dependent variable
    y = pdata['LN_QUANTITY']
    
    # Regressors
    X = pd.DataFrame({'const': 1, 'ln_price': pdata['LN_PRICE']})
    
    # Add standardized CPI control
    if pdata['FUEL_OIL_CPI'].notna().mean() > 0.5:
        cpi = pdata['FUEL_OIL_CPI'].fillna(pdata['FUEL_OIL_CPI'].mean())
        X['cpi_std'] = (cpi - cpi.mean()) / cpi.std()
    
    # Add standardized Industrial Production control
    if pdata['INDUSTRIAL_PRODUCTION'].notna().mean() > 0.5:
        ip = pdata['INDUSTRIAL_PRODUCTION'].fillna(pdata['INDUSTRIAL_PRODUCTION'].mean())
        X['ip_std'] = (ip - ip.mean()) / ip.std()
    
    # Month dummies for seasonality
    month_dummies = pd.get_dummies(pdata['MONTH'], prefix='m', drop_first=True)
    X = pd.concat([X.reset_index(drop=True), month_dummies.reset_index(drop=True)], axis=1)
    
    # Handle missing values
    mask = ~(X.isna().any(axis=1) | y.reset_index(drop=True).isna())
    X_clean = X[mask.values]
    y_clean = y.reset_index(drop=True)[mask.values]
    
    if len(y_clean) < min_obs:
        return None
    
    # Fit OLS
    try:
        model = OLS(y_clean, X_clean).fit()
        
        # Breusch-Pagan test for heteroskedasticity
        from statsmodels.stats.diagnostic import het_breuschpagan
        bp_stat, bp_pval, _, _ = het_breuschpagan(model.resid, X_clean)
        
        return {
            'product_id': product_id,
            'product_name': pdata['PRODUCT_NAME'].iloc[0],
            'product_family': pdata['PRODUCT_FAMILY'].iloc[0],
            'elasticity': model.params['ln_price'],
            'std_error': model.bse['ln_price'],
            't_stat': model.tvalues['ln_price'],
            'p_value': model.pvalues['ln_price'],
            'ci_lower': model.conf_int().loc['ln_price', 0],
            'ci_upper': model.conf_int().loc['ln_price', 1],
            'r_squared': model.rsquared,
            'adj_r_squared': model.rsquared_adj,
            'n_obs': int(model.nobs),
            'f_stat': model.fvalue,
            'f_pval': model.f_pvalue,
            'bp_stat': bp_stat,
            'bp_pval': bp_pval,
            'aic': model.aic,
            'bic': model.bic
        }
    except Exception as e:
        print(f'Error for {product_id}: {e}')
        return None

In [ ]:
# ============================================================================
# ESTIMATE ALL PRODUCTS
# ============================================================================
print('\n🔄 ESTIMATING OLS ELASTICITIES')
print('=' * 70)

results = []
for pid in df['PRODUCT_ID'].unique():
    result = estimate_ols_elasticity(df, pid)
    if result:
        results.append(result)
        sig = '***' if result['p_value'] < 0.01 else '**' if result['p_value'] < 0.05 else '*' if result['p_value'] < 0.1 else ''
        print(f"{result['product_name']:30s}  ε = {result['elasticity']:+7.3f} ({result['std_error']:.3f}) {sig:3s}  R² = {result['r_squared']:.3f}")

ols_df = pd.DataFrame(results)
print(f'\n✅ Successfully estimated: {len(ols_df)} products')

In [ ]:
# ============================================================================
# RESULTS VISUALIZATION
# ============================================================================
fig = go.Figure()

ols_sorted = ols_df.sort_values('elasticity')

# Add error bars for confidence intervals
fig.add_trace(go.Scatter(
    x=ols_sorted['elasticity'],
    y=ols_sorted['product_name'],
    mode='markers',
    marker=dict(size=10, color=ols_sorted['product_family'].astype('category').cat.codes, colorscale='Viridis'),
    error_x=dict(
        type='data',
        symmetric=False,
        array=ols_sorted['ci_upper'] - ols_sorted['elasticity'],
        arrayminus=ols_sorted['elasticity'] - ols_sorted['ci_lower']
    ),
    text=[f"ε={e:.3f} [{l:.3f}, {u:.3f}]" for e, l, u in zip(ols_sorted['elasticity'], ols_sorted['ci_lower'], ols_sorted['ci_upper'])],
    hoverinfo='text+y'
))

fig.add_vline(x=-1, line_dash='dash', line_color='white', annotation_text='Unit Elastic')
fig.add_vline(x=0, line_color='red', line_width=2)

fig.update_layout(
    title='<b>Own-Price Elasticity Estimates with 95% Confidence Intervals</b>',
    xaxis_title='Price Elasticity (ε)',
    template='plotly_dark',
    height=600,
    showlegend=False
)
fig.show()

## 4.2 Model Diagnostics

In [ ]:
# ============================================================================
# DIAGNOSTIC SUMMARY
# ============================================================================
print('\n📊 MODEL DIAGNOSTICS SUMMARY')
print('=' * 70)

n_negative = (ols_df['elasticity'] < 0).sum()
n_significant = (ols_df['p_value'] < 0.05).sum()
n_homoskedastic = (ols_df['bp_pval'] > 0.05).sum()

print(f"Negative elasticity (expected):     {n_negative:>3}/{len(ols_df)} ({n_negative/len(ols_df)*100:.1f}%)")
print(f"Significant at 5%:                  {n_significant:>3}/{len(ols_df)} ({n_significant/len(ols_df)*100:.1f}%)")
print(f"Pass homoskedasticity test (BP):    {n_homoskedastic:>3}/{len(ols_df)} ({n_homoskedastic/len(ols_df)*100:.1f}%)")
print(f"\nElasticity range: [{ols_df['elasticity'].min():.3f}, {ols_df['elasticity'].max():.3f}]")
print(f"Mean R²: {ols_df['r_squared'].mean():.3f}")
print(f"Median R²: {ols_df['r_squared'].median():.3f}")

---
<a id='sur'></a>
# 5. SUR Cross-Elasticity Model

## 5.1 Data Preparation

We aggregate to **product family level** for cleaner cross-elasticity estimation.

In [ ]:
# ============================================================================
# PREPARE SUR DATA
# ============================================================================
fam_df = session.sql('''
    SELECT 
        order_date,
        product_family,
        SUM(total_quantity_mt) as quantity,
        AVG(avg_price_per_mt) as price,
        AVG(fuel_oil_cpi) as cpi
    FROM CHEMICALS_DB.CHEMICAL_OPS.ML_TRAINING_DATA
    WHERE ln_quantity IS NOT NULL
    GROUP BY order_date, product_family
''').to_pandas()

families = sorted(fam_df['PRODUCT_FAMILY'].unique().tolist())
fam_names = [f.replace(' ', '_').upper() for f in families]
n_families = len(families)

print(f'\n📊 SUR DATA PREPARATION')
print(f'Families: {families}')
print(f'Observations per family: ~{len(fam_df) // n_families}')

# Pivot to wide format
price_wide = fam_df.pivot(index='ORDER_DATE', columns='PRODUCT_FAMILY', values='price').reset_index()
qty_wide = fam_df.pivot(index='ORDER_DATE', columns='PRODUCT_FAMILY', values='quantity').reset_index()
cpi_df = fam_df.groupby('ORDER_DATE')['CPI'].mean().reset_index()

sur_data = price_wide.merge(qty_wide, on='ORDER_DATE', suffixes=('_P', '_Q'))
sur_data = sur_data.merge(cpi_df, on='ORDER_DATE')

# Create log transforms
for fam, fname in zip(families, fam_names):
    sur_data[f'LN_P_{fname}'] = np.log(sur_data[f'{fam}_P'].clip(lower=1))
    sur_data[f'LN_Q_{fname}'] = np.log(sur_data[f'{fam}_Q'].clip(lower=1))

# Standardize CPI
sur_data['CPI_STD'] = (sur_data['CPI'] - sur_data['CPI'].mean()) / sur_data['CPI'].std()

sur_data = sur_data.dropna()
print(f'Clean observations: {len(sur_data)}')

## 5.2 SUR Estimation

In [ ]:
# ============================================================================
# ESTIMATE SUR SYSTEM
# ============================================================================
print('\n🔄 ESTIMATING SUR SYSTEM')
print('=' * 70)

price_cols = [f'LN_P_{f}' for f in fam_names]
sur_results = {}

# Estimate equation-by-equation (approximation to full SUR)
for fname in fam_names:
    y = sur_data[f'LN_Q_{fname}']
    X = sm.add_constant(sur_data[price_cols + ['CPI_STD']])
    
    model = OLS(y, X).fit()
    sur_results[fname] = model
    
    print(f'\n📊 {fname}')
    print(f'   R² = {model.rsquared:.4f}, Adj R² = {model.rsquared_adj:.4f}, N = {int(model.nobs)}')
    print(f'   F-stat = {model.fvalue:.2f} (p = {model.f_pvalue:.2e})')
    print('   Cross-elasticities:')
    for pcol in price_cols:
        pfname = pcol.replace('LN_P_', '')
        coef = model.params[pcol]
        se = model.bse[pcol]
        pval = model.pvalues[pcol]
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        own = '← OWN' if pfname == fname else ''
        print(f'      {pfname:15s}: {coef:+8.4f} ({se:.4f}) {sig:3s} {own}')

In [ ]:
# ============================================================================
# BUILD ELASTICITY MATRIX
# ============================================================================
E_matrix = pd.DataFrame(index=fam_names, columns=fam_names, dtype=float)
P_matrix = pd.DataFrame(index=fam_names, columns=fam_names, dtype=float)
SE_matrix = pd.DataFrame(index=fam_names, columns=fam_names, dtype=float)

for d_fam in fam_names:
    model = sur_results[d_fam]
    for p_fam in fam_names:
        col = f'LN_P_{p_fam}'
        E_matrix.loc[d_fam, p_fam] = model.params[col]
        P_matrix.loc[d_fam, p_fam] = model.pvalues[col]
        SE_matrix.loc[d_fam, p_fam] = model.bse[col]

print('\n📊 ELASTICITY MATRIX E = [εᵢⱼ]')
print('Row i: Demand for product i')
print('Col j: Price of product j')
print(E_matrix.round(4))

In [ ]:
# ============================================================================
# ELASTICITY MATRIX VISUALIZATION
# ============================================================================
fig = go.Figure()

# Heatmap
fig.add_trace(go.Heatmap(
    z=E_matrix.values.astype(float),
    x=[f.replace('_', ' ').title() for f in E_matrix.columns],
    y=[f.replace('_', ' ').title() for f in E_matrix.index],
    colorscale='RdBu',
    zmid=0,
    zmin=-2.5,
    zmax=1,
    text=E_matrix.round(3).values,
    texttemplate='%{text}',
    textfont={'size': 14, 'color': 'white'},
    hoverongaps=False,
    colorbar=dict(title='ε', titleside='right')
))

fig.update_layout(
    title='<b>Cross-Price Elasticity Matrix</b><br><sup>εᵢⱼ = ∂ln(Qᵢ)/∂ln(Pⱼ) | Diagonal = own-price | Off-diagonal positive = substitutes</sup>',
    xaxis_title='Price Change in Product j',
    yaxis_title='Demand for Product i',
    template='plotly_dark',
    height=500,
    width=600
)
fig.show()

## 5.3 Substitution Analysis

In [ ]:
# ============================================================================
# SUBSTITUTION PATTERN ANALYSIS
# ============================================================================
print('\n🔄 SUBSTITUTION PATTERN ANALYSIS')
print('=' * 70)

print('\n📊 OWN-PRICE ELASTICITIES (Diagonal)')
for i, fname in enumerate(fam_names):
    e = E_matrix.iloc[i, i]
    se = SE_matrix.iloc[i, i]
    classification = 'ELASTIC' if e < -1 else 'INELASTIC' if e > -1 else 'UNIT'
    print(f'   {fname:15s}: ε = {e:+.4f} (SE = {se:.4f}) → {classification}')

print('\n📊 SIGNIFICANT CROSS-ELASTICITIES (Off-diagonal, p < 0.05)')
for i, d_fam in enumerate(fam_names):
    for j, p_fam in enumerate(fam_names):
        if i != j:
            e = E_matrix.iloc[i, j]
            p = P_matrix.iloc[i, j]
            if p < 0.05:
                rel = 'SUBSTITUTE' if e > 0 else 'COMPLEMENT'
                print(f'   {d_fam} ← {p_fam}: ε = {e:+.4f} (p = {p:.4f}) → {rel}')
                print(f'      Interpretation: 1% ↑ in {p_fam} price → {e:.2f}% Δ in {d_fam} demand')

---
<a id='optimization'></a>
# 6. Optimization Engine

## 6.1 Mathematical Setup

In [ ]:
# ============================================================================
# OPTIMIZATION FUNCTIONS
# ============================================================================

def demand_function(P, Q0, E, P0):
    """
    Compute demand vector given prices using elasticity matrix.
    
    Q_i(P) = Q0_i * exp(Σ_j E_ij * ln(P_j / P0_j))
    
    Parameters:
    -----------
    P : np.ndarray, shape (n,)
        Price vector
    Q0 : np.ndarray, shape (n,)
        Baseline quantity vector
    E : np.ndarray, shape (n, n)
        Elasticity matrix
    P0 : np.ndarray, shape (n,)
        Reference price vector
    
    Returns:
    --------
    Q : np.ndarray, shape (n,)
        Predicted quantity vector
    """
    log_price_ratio = np.log(P / P0)
    log_Q_change = E @ log_price_ratio
    return Q0 * np.exp(log_Q_change)


def profit_function(P, Q0, C, E, P0):
    """
    Total contribution margin: Π(P) = Σ_i (P_i - C_i) * Q_i(P)
    """
    Q = demand_function(P, Q0, E, P0)
    return np.sum((P - C) * Q)


def profit_gradient(P, Q0, C, E, P0):
    """
    Gradient of profit function.
    
    ∂Π/∂P_k = Q_k + Σ_i (P_i - C_i) * Q_i * E_ik / P_k
    """
    Q = demand_function(P, Q0, E, P0)
    grad = np.zeros_like(P)
    for k in range(len(P)):
        grad[k] = Q[k] + np.sum((P - C) * Q * E[:, k]) / P[k]
    return grad


def negative_profit(P, Q0, C, E, P0):
    """Negative profit for minimization."""
    return -profit_function(P, Q0, C, E, P0)


def negative_gradient(P, Q0, C, E, P0):
    """Negative gradient for minimization."""
    return -profit_gradient(P, Q0, C, E, P0)

In [ ]:
# ============================================================================
# LOAD CURRENT STATE
# ============================================================================
current_df = session.sql('''
    SELECT 
        product_family,
        AVG(avg_price_per_mt) as current_price,
        AVG(avg_variable_cost) as variable_cost,
        SUM(total_quantity_mt) as total_quantity
    FROM CHEMICALS_DB.CHEMICAL_OPS.ML_TRAINING_DATA
    WHERE order_date >= DATEADD(day, -30, CURRENT_DATE())
    GROUP BY product_family
''').to_pandas()

current_df['FAMILY_ID'] = current_df['PRODUCT_FAMILY'].str.replace(' ', '_').str.upper()
current_df = current_df.set_index('FAMILY_ID').loc[fam_names]

P0 = current_df['CURRENT_PRICE'].values
Q0 = current_df['TOTAL_QUANTITY'].values
C = current_df['VARIABLE_COST'].values
E_np = E_matrix.values.astype(float)

print('\n📊 CURRENT STATE (Last 30 Days)')
print('=' * 70)
print(f'{"Product":<15} {"Price":>12} {"Cost":>12} {"Margin %":>10} {"Quantity":>12}')
print('-' * 70)
for i, fname in enumerate(fam_names):
    margin_pct = (P0[i] - C[i]) / P0[i] * 100
    print(f'{fname:<15} ${P0[i]:>11,.0f} ${C[i]:>11,.0f} {margin_pct:>9.1f}% {Q0[i]:>11,.0f}')

current_profit = profit_function(P0, Q0, C, E_np, P0)
current_revenue = np.sum(P0 * Q0)
print(f'\n💰 Current Total Profit: ${current_profit:,.0f}')
print(f'💵 Current Total Revenue: ${current_revenue:,.0f}')

## 6.2 Constraint Specification

| Constraint | Formula | Business Meaning |
|------------|---------|------------------|
| Margin floor | $(P_i - C_i)/P_i \geq m^{min}$ | Minimum 15% gross margin |
| Price ceiling | $P_i \leq P_i^0 (1 + \delta^+)$ | Max 10% price increase |
| Price floor | $P_i \geq P_i^0 (1 - \delta^-)$ | Max 10% price decrease |

In [ ]:
# ============================================================================
# CONSTRAINT CONFIGURATION
# ============================================================================
MIN_MARGIN = 0.15      # 15% minimum margin
MAX_INCREASE = 0.10    # 10% max price increase
MAX_DECREASE = 0.10    # 10% max price decrease

# Build bounds
bounds = []
for i in range(len(P0)):
    # Lower bound: max of (price floor, margin floor)
    floor_from_change = P0[i] * (1 - MAX_DECREASE)
    floor_from_margin = C[i] / (1 - MIN_MARGIN)
    lower = max(floor_from_change, floor_from_margin)
    
    # Upper bound: price ceiling
    upper = P0[i] * (1 + MAX_INCREASE)
    
    bounds.append((lower, upper))

print('\n⚙️ CONSTRAINT CONFIGURATION')
print('=' * 70)
print(f'Minimum Margin: {MIN_MARGIN*100:.0f}%')
print(f'Max Price Increase: +{MAX_INCREASE*100:.0f}%')
print(f'Max Price Decrease: -{MAX_DECREASE*100:.0f}%')
print(f'\n{"Product":<15} {"Lower Bound":>12} {"Upper Bound":>12} {"Current":>12}')
print('-' * 55)
for i, fname in enumerate(fam_names):
    print(f'{fname:<15} ${bounds[i][0]:>11,.0f} ${bounds[i][1]:>11,.0f} ${P0[i]:>11,.0f}')

In [ ]:
# ============================================================================
# RUN OPTIMIZATION
# ============================================================================
print('\n🚀 RUNNING OPTIMIZATION')
print('=' * 70)

result = minimize(
    negative_profit,
    x0=P0,
    args=(Q0, C, E_np, P0),
    method='SLSQP',
    jac=negative_gradient,
    bounds=bounds,
    options={'maxiter': 1000, 'ftol': 1e-10, 'disp': True}
)

P_opt = result.x
Q_opt = demand_function(P_opt, Q0, E_np, P0)
optimal_profit = profit_function(P_opt, Q0, C, E_np, P0)
optimal_revenue = np.sum(P_opt * Q_opt)

print(f'\nSolver Status: {"✅ CONVERGED" if result.success else "❌ FAILED"}')
print(f'Iterations: {result.nit}')
print(f'Final Objective: {-result.fun:,.0f}')

## 6.3 Optimization Results

In [ ]:
# ============================================================================
# RESULTS TABLE
# ============================================================================
print('\n📊 OPTIMIZATION RESULTS')
print('=' * 90)
print(f'{"Product":<15} {"Current":>10} {"Optimal":>10} {"Δ Price":>10} {"Δ Volume":>10} {"Δ Margin":>12} {"Status":<12}')
print('-' * 90)

total_margin_change = 0
for i, fname in enumerate(fam_names):
    price_change_pct = (P_opt[i] - P0[i]) / P0[i] * 100
    volume_change_pct = (Q_opt[i] - Q0[i]) / Q0[i] * 100
    current_margin = (P0[i] - C[i]) * Q0[i]
    optimal_margin = (P_opt[i] - C[i]) * Q_opt[i]
    margin_change = optimal_margin - current_margin
    total_margin_change += margin_change
    
    # Check binding constraints
    at_floor = abs(P_opt[i] - bounds[i][0]) < 1
    at_ceiling = abs(P_opt[i] - bounds[i][1]) < 1
    status = '📍 FLOOR' if at_floor else ('📍 CEILING' if at_ceiling else '✓')
    
    print(f'{fname:<15} ${P0[i]:>9,.0f} ${P_opt[i]:>9,.0f} {price_change_pct:>+9.1f}% {volume_change_pct:>+9.1f}% ${margin_change:>+11,.0f} {status}')

print('-' * 90)
profit_improvement = (optimal_profit - current_profit) / current_profit * 100
revenue_change = (optimal_revenue - current_revenue) / current_revenue * 100
print(f'{"TOTAL":<15} {"":>10} {"":>10} {"":>10} {"":>10} ${total_margin_change:>+11,.0f}')

print(f'\n💰 PROFIT SUMMARY')
print(f'   Current:    ${current_profit:>15,.0f}')
print(f'   Optimal:    ${optimal_profit:>15,.0f}')
print(f'   Improvement: ${optimal_profit - current_profit:>+14,.0f} ({profit_improvement:+.1f}%)')

In [ ]:
# ============================================================================
# RESULTS VISUALIZATION
# ============================================================================
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Price Changes (%)', 'Volume Impact (%)', 'Margin Waterfall ($)', 'Current vs Optimal Price'],
    specs=[[{}, {}], [{'colspan': 2}, None]]
)

# Price changes
price_changes = [(P_opt[i] - P0[i]) / P0[i] * 100 for i in range(len(P0))]
colors = ['#22c55e' if c >= 0 else '#ef4444' for c in price_changes]
fig.add_trace(go.Bar(
    x=[f.replace('_', ' ').title() for f in fam_names], y=price_changes,
    marker_color=colors, text=[f'{c:+.1f}%' for c in price_changes], textposition='outside'
), row=1, col=1)

# Volume changes
volume_changes = [(Q_opt[i] - Q0[i]) / Q0[i] * 100 for i in range(len(Q0))]
vol_colors = ['#3b82f6' if c >= 0 else '#f59e0b' for c in volume_changes]
fig.add_trace(go.Bar(
    x=[f.replace('_', ' ').title() for f in fam_names], y=volume_changes,
    marker_color=vol_colors, text=[f'{c:+.1f}%' for c in volume_changes], textposition='outside'
), row=1, col=2)

# Margin waterfall
margin_changes = [(P_opt[i]-C[i])*Q_opt[i] - (P0[i]-C[i])*Q0[i] for i in range(len(P0))]
fig.add_trace(go.Waterfall(
    x=[f.replace('_', ' ').title() for f in fam_names] + ['TOTAL'],
    y=margin_changes + [sum(margin_changes)],
    measure=['relative']*len(margin_changes) + ['total'],
    text=[f'${x:+,.0f}' for x in margin_changes + [sum(margin_changes)]],
    textposition='outside',
    connector={'line': {'color': 'rgba(255,255,255,0.3)'}},
    increasing={'marker': {'color': '#22c55e'}},
    decreasing={'marker': {'color': '#ef4444'}},
    totals={'marker': {'color': '#3b82f6'}}
), row=2, col=1)

fig.update_layout(template='plotly_dark', height=700, showlegend=False,
                  title='<b>Optimization Results Dashboard</b>')
fig.show()

---
<a id='results'></a>
# 7. Executive Summary & Recommendations

In [ ]:
# ============================================================================
# EXECUTIVE SUMMARY
# ============================================================================
print('\n' + '=' * 70)
print('📋 EXECUTIVE SUMMARY: CHEMICALS PRICING OPTIMIZATION')
print('=' * 70)

print('\n1️⃣  OWN-PRICE ELASTICITIES')
print('-' * 50)
for fname in fam_names:
    e = E_matrix.loc[fname, fname]
    status = 'ELASTIC (↑P → ↓Rev)' if e < -1 else 'INELASTIC (↑P → ↑Rev)'
    print(f'   {fname:15s}: ε = {e:+.3f} → {status}')

print('\n2️⃣  KEY SUBSTITUTION PATTERNS')
print('-' * 50)
for i, d in enumerate(fam_names):
    for j, p in enumerate(fam_names):
        if i != j and E_matrix.iloc[i, j] > 0.1 and P_matrix.iloc[i, j] < 0.05:
            e = E_matrix.iloc[i, j]
            print(f'   {d} ← {p}: ε = {e:+.3f}')
            print(f'      → If {p} ↑10%, {d} demand ↑{e*10:.1f}%')

print('\n3️⃣  OPTIMIZATION IMPACT')
print('-' * 50)
print(f'   💰 Profit Improvement: +${optimal_profit - current_profit:,.0f} ({profit_improvement:+.1f}%)')
print(f'   📊 Within constraints: ±{MAX_INCREASE*100:.0f}% price, {MIN_MARGIN*100:.0f}% min margin')

print('\n4️⃣  RECOMMENDED ACTIONS')
print('-' * 50)
for i, fname in enumerate(fam_names):
    change = (P_opt[i] - P0[i]) / P0[i] * 100
    if change > 2:
        action = f'↑ INCREASE by {change:.1f}%'
    elif change < -2:
        action = f'↓ DECREASE by {abs(change):.1f}%'
    else:
        action = '→ HOLD current price'
    print(f'   {fname:15s}: {action}')

print('\n' + '=' * 70)
print('✅ ANALYSIS COMPLETE')
print('=' * 70)

---

## Appendix: Technical Notes

### A. Model Assumptions & Limitations

| Assumption | Implication | Mitigation |
|------------|-------------|------------|
| Price exogeneity | OLS consistent | Use IV/2SLS if violated |
| Constant elasticity | May not hold at extremes | Validate in-sample |
| No competitive response | Ignores game theory | Layer in response models |
| Static optimization | Single-period | Extend to dynamic pricing |

### B. References

1. Zellner, A. (1962). "An Efficient Method of Estimating Seemingly Unrelated Regressions." *JASA*.
2. Nevo, A. (2000). "A Practitioner's Guide to Estimation of Random-Coefficients Logit Models." *J. Econ. & Mgmt. Strategy*.
3. Talluri, K. & van Ryzin, G. (2004). *The Theory and Practice of Revenue Management*. Springer.